# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [ ]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


Quadro RTX 5000, 16384 MiB, 16110 MiB
Deps installed


In [ ]:
!pip install "transformers==4.43.4" "sentence-transformers==2.7.0" "huggingface-hub==0.25.0" "accelerate==0.33.0" "peft==0.13.2" "bitsandbytes==0.43.3" -q


In [ ]:
# Cell 2: Clone CogMem + load tasks
REPO_BRANCH = "master"
!if [ ! -d /notebooks/CogMem/.git ]; then     git clone --branch {REPO_BRANCH} --single-branch https://github.com/tungooxx/CogMem.git /notebooks/CogMem; else     cd /notebooks/CogMem && git fetch origin && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}; fi
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for task in tasks:
            f.write(json.dumps(task) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print(Path('/notebooks/CogMem').resolve())
!cd /notebooks/CogMem && git branch --show-current && git rev-parse --short HEAD
print("Tasks:", len(tasks))
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


Already on 'feat/episode-cluster-memories'
Your branch is up to date with 'origin/feat/episode-cluster-memories'.
From https://github.com/tungooxx/CogMem
 * branch            feat/episode-cluster-memories -> FETCH_HEAD
Already up to date.
feat/episode-cluster-memories
328b8aa
Tasks: 1140
Train (create patches): 200
Eval (test patches): 940


In [6]:
# Cell 2b: Verify package runner import
import inspect
import cogmem.patches.experiment as patch_experiment

print("experiment loaded from:", patch_experiment.__file__)
print("runner functions:")
for name in [
    "load_patch_runtime",
    "run_patch_episode_recording",
    "build_patch_memories",
    "inspect_unseen_retrieval",
    "sweep_patch_retrieval_width",
    "evaluate_patch_memory_bank",
    "summarize_patch_results",
]:
    print(f"  {name}: {hasattr(patch_experiment, name)}")


loaded from: /notebooks/CogMem/cogmem/patches/memory_bank.py
has transfer selector: True
contains filtered token: True
def _extract_structural_markers(
    prompts: list[str],
    negative_prompts: list[str] | None = None,
    max_markers: int = DEFAULT_RETRIEVE_MARKERS,
) -> list[str]:
    positive_counts: dict[str, int] = {}
    for prompt in prompts:
        for token in set(_prompt_feature_tokens(prompt)):
            positive_counts[token] = positive_counts.get(token, 0) + 1
    negative_counts: dict[str, int] = {}
    for prompt in negative_prompts or []:
        for token in set(_prompt_feature_tokens(prompt)):
            negative_counts[token] = negative_counts.get(token, 0) + 1
    positive_total = max(len(prompts), 1)
    negative_total = max(len(negative_prompts or []), 1)
    ranked: list[tuple[float, int, float, str]] = []
    for token, positive_count in positive_counts.items():
        positive_rate = positive_count / positive_total
        negative_rate = negative_coun

In [7]:
# Cell 3: Load patch experiment runtime + config
import torch
from cogmem.patches.experiment import (
    PatchExperimentConfig,
    load_patch_runtime,
    run_patch_episode_recording,
    build_patch_memories,
    inspect_unseen_retrieval,
    sweep_patch_retrieval_width,
    evaluate_patch_memory_bank,
    summarize_patch_results,
)
from cogmem.patches.memory_bank import ClusterMemoryBank

RESET_CELL4_PROGRESS = False
FORCE_RERUN_EVAL = False

PATCH_CONFIG = PatchExperimentConfig(
    memory_dir="/notebooks/cogmem_cluster_memories",
    train_task_count=len(TRAIN_TASKS),
    cluster_similarity_threshold=0.62,
    cluster_min_support=3,
    cluster_control_episodes=6,
    inspect_unseen_size=200,
    sweep_diag_size=30,
    sweep_topk_options=(1, 2, 5),
    eval_top_k=1,
    eval_scale=0.25,
    eval_cache_version="finaluse_v4",
    unseen_eval_size=50,
)

print("Patch config:")
for key, value in PATCH_CONFIG.__dict__.items():
    print(f"  {key}: {value}")

print()
print("Loading model + embedder...")
base_model, tokenizer, embedder = load_patch_runtime(
    model_name=PATCH_CONFIG.model_name,
    embedder_name=PATCH_CONFIG.embedder_name,
    prepare_for_training=True,
)
free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Runtime loaded. Free VRAM: {free:.1f} GB")


Loading model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 4: Record episodes from TRAIN_TASKS
memory_bank = ClusterMemoryBank(PATCH_CONFIG.memory_dir)
memory_bank.load()

collection_result = run_patch_episode_recording(
    TRAIN_TASKS,
    base_model,
    tokenizer,
    embedder,
    config=PATCH_CONFIG,
    memory_bank=memory_bank,
    reset_progress=RESET_CELL4_PROGRESS,
    verbose=True,
)

print()
print("=" * 50)
print("EPISODE RECORDING COMPLETE")
print("Tasks processed:", collection_result["tasks_processed"])
print("Tasks with passes:", collection_result["tasks_with_passes"])
print("New episodes recorded:", collection_result["new_episodes"])
print("Episodes total:", collection_result["episodes_total"])
print("Time:", round(collection_result["elapsed_minutes"], 1), "min")
print("Progress file:", collection_result["progress_path"])
print("Memory bank stats:", collection_result["stats"])


Base model prepared for training
[1/200] BigCodeBench/0: 0P/4F | episodes=0 | pass_rate=0/1 | 99/hr
[2/200] BigCodeBench/1: 4P/0F | episodes=0 | pass_rate=1/2 | 112/hr
[3/200] BigCodeBench/2: 0P/4F | episodes=0 | pass_rate=1/3 | 113/hr
[4/200] BigCodeBench/3: 2P/2F | episodes=1 | pass_rate=2/4 | 112/hr
[5/200] BigCodeBench/4: 2P/2F | episodes=2 | pass_rate=3/5 | 119/hr
[10/200] BigCodeBench/9: 4P/0F | episodes=3 | pass_rate=6/10 | 104/hr
[20/200] BigCodeBench/19: 0P/4F | episodes=5 | pass_rate=8/20 | 79/hr
[30/200] BigCodeBench/29: 1P/3F | episodes=9 | pass_rate=15/30 | 85/hr
[40/200] BigCodeBench/39: 0P/4F | episodes=11 | pass_rate=18/40 | 83/hr
[50/200] BigCodeBench/49: 0P/4F | episodes=13 | pass_rate=20/50 | 81/hr
[60/200] BigCodeBench/59: 0P/4F | episodes=17 | pass_rate=25/60 | 82/hr
[70/200] BigCodeBench/69: 3P/1F | episodes=21 | pass_rate=29/70 | 76/hr
[80/200] BigCodeBench/79: 0P/4F | episodes=22 | pass_rate=30/80 | 75/hr
[90/200] BigCodeBench/89: 0P/4F | episodes=24 | pass_rate

In [ ]:
# Cell 4b: Build cluster memories and inspect distilled artifacts
memory_bank = ClusterMemoryBank(PATCH_CONFIG.memory_dir)
memory_bank.load()
print("Loaded episodes:", len(memory_bank.episodes))

if len(memory_bank.episodes) == 0:
    raise RuntimeError(
        "No saved episodes found in /notebooks/cogmem_cluster_memories. "
        "Run Cell 4 to completion before building cluster memories."
    )

print("Cluster settings:", {
    "similarity_threshold": PATCH_CONFIG.cluster_similarity_threshold,
    "min_support": PATCH_CONFIG.cluster_min_support,
    "control_episodes": PATCH_CONFIG.cluster_control_episodes,
})

build_result = build_patch_memories(
    base_model,
    tokenizer,
    memory_bank,
    config=PATCH_CONFIG,
    eval_tasks=EVAL_TASKS,
    embedder=embedder,
)
print("Memory bank stats:", build_result["build_stats"])
print("Retrievable memories:", build_result["retrievable_memories"])

print("=== CLUSTER MEMORY SUMMARY ===")
for row in build_result["memory_rows"]:
    print("Memory:", row["memory_id"])
    print(
        "  family:", row["family"],
        "support:", row["support_count"],
        "promote:", round(row["promotion_score"], 3),
        "threshold(app):", round(row["threshold"], 3),
        "retrievable:", row["retrievable"],
    )
    print(
        "  local_gain:", round(row["local_gain"], 4),
        "heldout_gain:", round(row["heldout_gain"], 4),
        "transfer_gain:", round(row["transfer_gain"], 4),
        "transfer_online_gain:", round(row["transfer_online_gain"], 4),
        "transfer_rate:", round(row["transfer_rate"], 3),
    )
    print(
        "  recent_success:", round(row["recent_success_rate"], 3),
        "online_hurt:", round(row["online_hurt_rate"], 3),
        "utility_regression:", round(row["utility_regression"], 4),
        "redundancy:", round(row["redundancy_penalty"], 4),
    )
    print(
        "  neg_penalty:", round(row["negative_steering_penalty"], 4),
        "negatives:", row["negative_count"],
        "markers:", row["structural_markers"],
    )
    print("  patches:", row["patch_ids"])

if not build_result["artifact_rows"]:
    print("No retrievable memories yet. Add more episodes or inspect family clustering.")
else:
    print("
[1] Distilled artifact magnitudes:")
    for row in build_result["artifact_rows"]:
        if not row.get("patch_id"):
            print(f"  {row['memory_id']}: no artifact patch loaded")
            continue
        print(
            f"  {row['memory_id'][:36]} -> {row['patch_id'][:36]}: "
            f"|A|={row['norm_a']:.4f} |B|={row['norm_b']:.4f} avg_norm={row['avg_norm']:.4f}"
        )

print("
[2] Output difference (first 3 eval tasks):")
for row in build_result["output_rows"]:
    if row.get("abstained"):
        print(f"  {row['task_id']}: ABSTAINED")
        continue
    print(
        f"  {row['task_id']}: {row['token_diff_rate'] * 100:.0f}% tokens different | "
        f"memory={row['memory_id']} | final_use={row['final_use']:.3f} | "
        f"use={row['q_use']:.3f} | app={row['applicability']:.3f} | "
        f"threshold={row['threshold']:.3f} | promote={row['q_promote']:.3f}"
    )


Loaded 0 patches (0 promoted) from /notebooks/cogmem_cluster_memories/patch_artifacts
Loaded episodes: 52


max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs


Memory bank stats: {'episodes': 52, 'memories': 7, 'retrievable_memories': 7, 'artifact_patches': 7, 'mean_promotion': 0.6725639935263213, 'mean_q': 0.6725639935263213, 'families': {'random_numeric': 1, 'networking': 1, 'file_io': 1, 'dataframe': 3, 'plotting': 1}}
Retrievable memories: 7
=== CLUSTER MEMORY SUMMARY ===
Memory: memory_random_numeric_0845eb44ce
  family: random_numeric support: 6 promote: 0.719 threshold(app): 0.365 retrievable: True
  local_gain: 1.5386 heldout_gain: 1.2594 transfer_gain: 1.2594 transfer_online_gain: 0.0 transfer_rate: 1.0
  recent_success: 0.0 online_hurt: 0.0 utility_regression: 0.0 redundancy: 0.0
  neg_penalty: 0.0 negatives: 3 markers: ['contained', 'import', 'random', 'starting', 'task_func']
  payload keys: ['cluster_metadata', 'evidence', 'transfer_stats', 'patch_ids']
  patches: ['cluster_patch_memory_random_numeric_0845eb44ce']
Memory: memory_networking_84e1653036
  family: networking support: 3 promote: 0.702 threshold(app): 0.358 retrievable

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  BigCodeBench/200: 74% tokens different | memory=memory_plotting_461f4c79e9 | final_use=0.373 | use=0.262 | app=0.403 | threshold=0.381 | promote=0.740
  BigCodeBench/201: ABSTAINED
  BigCodeBench/202: ABSTAINED


In [ ]:
# Cell 4b+: Quick memory bank stats
memory_bank = ClusterMemoryBank(PATCH_CONFIG.memory_dir)
memory_bank.load()
for key, value in memory_bank.stats().items():
    print(f"{key}: {value}")


2026-04-16 09:31:52.026136: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 09:31:52.026213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 09:31:52.027836: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 09:31:52.035744: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 09:31:53.248988: W tensorflow/compiler/tf2

Loaded 7 patches (0 promoted) from /notebooks/cogmem_cluster_memories/patch_artifacts
Episodes: 52
Memories: 7
Artifact patches: 7


In [ ]:
# Cell 4c: Sample package-runner output deltas
if not build_result.get("output_rows"):
    print("No sampled output rows available. Run Cell 4b first.")
else:
    for row in build_result["output_rows"]:
        if row.get("abstained"):
            print(f"{row['task_id']}: ABSTAINED")
        else:
            print(
                f"{row['task_id']} | memory={row['memory_id']} | "
                f"final_use={row['final_use']:.3f} | use={row['q_use']:.3f} | "
                f"app={row['applicability']:.3f} | threshold={row['threshold']:.3f} | "
                f"promote={row['q_promote']:.3f}"
            )


/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Selected memories: ['memory_plotting_461f4c79e9']
Artifact patches: ['cluster_patch_memory_plotting_461f4c79e9']
Applicability: 0.4033
Q_use      : 0.2622
FinalUse   : 0.3732
Q_promote  : 0.74
Threshold  : 0.3814
Payload keys: ['cluster_metadata', 'evidence', 'transfer_stats', 'patch_ids']

=== Greedy (temp=0) ===
Cold first 100: Thought: 
To solve this task, I need to generate 'n' random numbers between 0 and 1. Then, I'll calc
Patched first 100: import random
import bisect
import statistics
import matplotlib.pyplot as plt

def task_func(n, valu
IDENTICAL: False

=== Hook Count ===
Hooks registered: 144
Hooks removed: True

=== Low-temp (0.01) ===
IDENTICAL: False


In [ ]:
# Cell 5: Inspect applicability + final use on the first unseen tasks
unseen_result = inspect_unseen_retrieval(
    EVAL_TASKS,
    memory_bank,
    embedder,
    config=PATCH_CONFIG,
)

print("Unseen tasks inspected:", unseen_result["unseen_tasks_inspected"])
print("Retrievable memories:", unseen_result["retrievable_memories"])
print("Mean top-1 final use: {:.3f}".format(unseen_result["mean_top1_final_use"]))
print("Mean top-1 Q_use: {:.3f}".format(unseen_result["mean_top1_q_use"]))
print("Mean top-1 applicability: {:.3f}".format(unseen_result["mean_top1_applicability"]))
print("Mean applicability margin: {:.3f}".format(unseen_result["mean_applicability_margin"]))
print(
    "Abstentions: {}/{} ({:.1%})".format(
        unseen_result["abstentions"],
        unseen_result["unseen_tasks_inspected"],
        unseen_result["abstention_rate"],
    )
)

print()
print("Most selected memories (after gate):")
for memory_id, hits in sorted(unseen_result["memory_hits"].items(), key=lambda item: item[1], reverse=True):
    print(f"  {memory_id} -> {hits} tasks")

print()
print("Most selected families (after gate):")
for family, hits in sorted(unseen_result["family_hits"].items(), key=lambda item: item[1], reverse=True):
    print(f"  {family} -> {hits} tasks")

print()
print("Top unseen examples:")
for row in unseen_result["rows"][:10]:
    print(
        "  {task_id} | final_use={final_use:.3f} | use={q_use:.3f} | app={applicability:.3f} | "
        "memory={memory_id} | family={family} | selected={selected}".format(**row)
    )


Unseen tasks inspected: 200
Retrievable memories: 7
Mean top-1 final use: 0.365
Mean top-1 Q_use: 0.256
Mean top-1 applicability: 0.394
Mean applicability margin: 0.037
Abstentions: 107/200 (53.5%)

Most selected memories (after gate):
  memory_file_io_040cb3229f -> 37 tasks
  memory_plotting_461f4c79e9 -> 35 tasks
  memory_random_numeric_0845eb44ce -> 12 tasks
  memory_dataframe_2b3ba718c6 -> 7 tasks
  memory_dataframe_d144959b3a -> 1 tasks
  memory_networking_84e1653036 -> 1 tasks

Most selected families (after gate):
  file_io -> 37 tasks
  plotting -> 35 tasks
  random_numeric -> 12 tasks
  dataframe -> 8 tasks
  networking -> 1 tasks

Top unseen examples:
  BigCodeBench/200 | final_use=0.373 | use=0.262 | app=0.403 | memory=memory_plotting_461f4c79e9 | family=plotting | selected=True
  BigCodeBench/201 | final_use=0.000 | use=0.000 | app=0.000 | memory=ABSTAIN | family= | selected=False
  BigCodeBench/202 | final_use=0.000 | use=0.000 | app=0.000 | memory=ABSTAIN | family= | selec

In [ ]:
# Cell 5b: Sweep retrieval width on a small seen slice
sweep_results = sweep_patch_retrieval_width(
    TRAIN_TASKS,
    base_model,
    tokenizer,
    memory_bank,
    embedder,
    config=PATCH_CONFIG,
    verbose=True,
)

print()
print("{:<5} {:>8} {:>8} {:>8} {:>8} {:>8}".format("k", "cold", "memory", "delta", "hurt", "abstain"))
print("-" * 64)
for row in sweep_results:
    print(
        "{:<5} {:>7.1%} {:>7.1%} {:>+7.1%} {:>8} {:>8}".format(
            row["top_k"],
            row["cold_rate"],
            row["memory_rate"],
            row["delta"],
            row["hurt"],
            row["abstained"],
        )
    )
print()
print("Best setting:", sweep_results[0] if sweep_results else None)


Running gated retrieval sweep on 30 seen tasks


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  cached cold [10/30] cold=4
  cached cold [20/30] cold=4
  cached cold [30/30] cold=8

-- top_k=1, scale=0.25 --


/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

-- top_k=2, scale=0.25 --
  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

-- top_k=5, scale=0.25 --
  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

k         cold   memory    delta     hurt  abstain
----------------------------------------------------------------
1       26.7%   40.0%  +13.3%        0       21
2       26.7%   40.0%  +13.3%        0       21
5       26.7%   40.0%  +13.3%        0       21

Best setting: {'top_k': 1, 'cold_passed': 8, 'memory_passed': 12, 'cold_rate': 0.26666666666666666, 'memory_rate': 0.4, 'delta': 0.13333333333333333, 'helped': 4, 'hurt': 0, 'abstained': 21}


In [ ]:
# Cell 6: Evaluate gated cluster memories on seen and unseen tasks
print("Seen tasks:", len(TRAIN_TASKS))
print("Unseen tasks:", min(PATCH_CONFIG.unseen_eval_size, len(EVAL_TASKS)))
print("Episodes:", len(memory_bank.episodes))
print("Memories:", len(memory_bank.memories))
print("Artifact patches:", len(memory_bank.artifact_bank.patches))

evaluation_result = evaluate_patch_memory_bank(
    TRAIN_TASKS,
    EVAL_TASKS,
    base_model,
    tokenizer,
    memory_bank,
    embedder,
    config=PATCH_CONFIG,
    force_rerun=FORCE_RERUN_EVAL,
    verbose=True,
)
seen_eval = evaluation_result["seen_eval"]
unseen_eval = evaluation_result["unseen_eval"]
print("Eval cache:", evaluation_result["cache_path"])


Seen tasks: 200
Unseen tasks: 50
Episodes: 52
Memories: 7
Artifact patches: 7
Eval cache: /notebooks/cogmem_cluster_memories/eval_cache_finaluse_v4_seen200_unseen50_top5_scale0p25.json

--- SEEN COLD + MEMORY EVAL ---
  [50/200] cold: 10/50 (20.0%) | memory: 14/50 (28.0%) | abstain=30
  [100/200] cold: 24/100 (24.0%) | memory: 25/100 (25.0%) | abstain=72
  [150/200] cold: 39/150 (26.0%) | memory: 37/150 (24.7%) | abstain=109
  [200/200] cold: 51/200 (25.5%) | memory: 50/200 (25.0%) | abstain=155
SEEN cold result: 51 / 200 (25.5%)
SEEN memory result: 50 / 200 (25.0%)
SEEN memory usage: used=45 abstained=155 (77.5% abstain)

--- UNSEEN COLD + MEMORY EVAL ---
  [50/50] cold: 9/50 (18.0%) | memory: 9/50 (18.0%) | abstain=50
UNSEEN cold result: 9 / 50 (18.0%)
UNSEEN memory result: 9 / 50 (18.0%)
UNSEEN memory usage: used=0 abstained=50 (100.0% abstain)
Saved eval cache to /notebooks/cogmem_cluster_memories/eval_cache_finaluse_v4_seen200_unseen50_top5_scale0p25.json


In [ ]:
# Cell 7: Results comparison + current score formulas
summary_result = summarize_patch_results(
    memory_bank,
    seen_eval,
    unseen_eval,
    config=PATCH_CONFIG,
)

print("=" * 72)
print("EPISODE-FIRST CLUSTER MEMORY RESULTS")
print("=" * 72)
print()
print("Episodes recorded:", summary_result["episodes"])
print("Cluster memories built:", summary_result["memories"])
print("Artifact patches available:", summary_result["artifact_patches"])
print("Eval top_k:", summary_result["eval_top_k"], "| Eval scale:", summary_result["eval_scale"])
print()
print("{:<12} {:>8} {:>8} {:>9} {:>9} {:>9} {:>10}".format("Split", "Cold", "Memory", "Cold %", "Mem %", "Delta", "Abstain"))
print("-" * 72)
for result in [summary_result["seen_eval"], summary_result["unseen_eval"]]:
    print(
        "{:<12} {:>8} {:>8} {:>8.1%} {:>8.1%} {:>+8.1%} {:>9.1%}".format(
            result["label"],
            result["cold_passed"],
            result["memory_passed"],
            result["cold_rate"],
            result["memory_rate"],
            result["delta"],
            result["abstained"] / max(result["total"], 1),
        )
    )

print()
print("Current wake retrieval:")
print("  applicability =", summary_result["current_formulas"]["applicability"])
print("  Q_use =", summary_result["current_formulas"]["q_use"])
print("  FinalUse =", summary_result["current_formulas"]["final_use"])
print("  drop memories with applicability <= retrieval_threshold")
print("  use top-1 unless FinalUse is very high and nearby memories stay within a small margin")
print()
print("Current sleep promotion:")
print("  Q_promote =", summary_result["current_formulas"]["q_promote"])
print("  promote if Q_promote >= 0.40 and support_count >= 3")
print("  demote if preserve set is harmed")
print("  prune if Q_promote <= 0.10, support_count < 3, and preserve set is harmed")
print("  legacy q_value mirrors promotion_score for compatibility")

print()
if summary_result["unseen_eval"]["delta"] > 0.01:
    print("Unseen-task memory improvement is positive.")
elif summary_result["unseen_eval"]["delta"] > -0.01:
    print("Unseen-task memory effect is roughly neutral.")
else:
    print("Unseen-task memory effect is negative.")

print()
print("Memory bank:")
for key, value in summary_result["memory_stats"].items():
    print(f"  {key}: {value}")


EPISODE-FIRST CLUSTER MEMORY RESULTS

Episodes recorded: 52
Cluster memories built: 7
Artifact patches available: 7
Eval top_k: 5 | Eval scale: 0.25

Split            Cold   Memory    Cold %     Mem %     Delta    Abstain
------------------------------------------------------------------------
SEEN               51       50    25.5%    25.0%    -0.5%     77.5%
UNSEEN              9        9    18.0%    18.0%    +0.0%    100.0%

Current wake retrieval:
  applicability = clip(0.60 * pos_sim - 0.25 * neg_sim + 0.15 * structural_match - 0.10 * hard_negative_margin_penalty, 0, 1)
  Q_use = applicability * clip(0.45 * transfer_gain + 0.20 * recent_success_rate + 0.15 * log_reuse - 0.20 * online_hurt_rate, 0, 1)
  FinalUse = clip(Q_use + 0.15 * Q_promote, 0, 1)
  drop memories with applicability <= retrieval_threshold
  use top-1 unless FinalUse is very high and nearby memories stay within a small margin

Current sleep promotion:
  Q_promote = 0.28 * heldout_gain + 0.16 * transfer_gain + 0.06